# C1DB1B: Data Management for Decision-Making
## Workshop 2: Data Preparation, Hygiene & Pipeline Integration
### Executive decision: Where should Vårdlyftet establish its next primary care centre?

---

### The assignment you are supporting
Vårdlyftet AB is considering a primary care centre in Västra Götaland. Your group has been allocated **five municipalities**. The management question, from Document A (*The Assignment*), is:

> Which municipality in your portfolio should Vårdlyftet choose for its next primary care centre, **if any**, and why?

You are advising Vårdlyftet. You are **not** making a regional policy decision on behalf of Västra Götalandsregionen.

In **Module 2 you still do not pick a winner**. You must prepare evidence that can be compared consistently:

1. confirm what each source is (observation unit, keys, coverage, missing markers);
2. join sources without losing alternatives or inventing values;
3. audit missingness and compare two defensible treatments;
4. keep original indicator meaning unless a transformation has a stated purpose;
5. produce a five-row decision table and respond to a controlled data defect.

Keep this question in view throughout the notebook:

> How does the work we are doing now help us decide where Vårdlyftet should establish its next primary care centre?

---

### Sequence for this module

**Input Contract** → **Integration** → **Diagnosis** → **Treatment** → **Verified Output**

Complete the **common companion exercise** first so the method is visible. Then join **every** additional source named in your approved KPI-to-data table. Those group extracts belong in `data/group_sources/`. They are the rest of your architecture, not optional extras. There is no fixed dataset count and no requirement to use different transformations merely to demonstrate variety.

---

### Data vintage (do not hard-code years or file names)
This notebook reads vintage, paths, reference years, and the predictive-component flag from `data/manifest.json`.

Current contract (Document B, *The Data Stream*):
- Module 1 anchor: `data/01_module1/VGR_Demographics_Anchor.csv`
- Annual companion: `data/02_module2/VGR_Companion_Annual.csv` (one municipality × year $t_1$)
- Static lookup: `data/02_module2/VGR_Companion_StaticLookup.csv` (one municipality; **no year**)
- Masked teaching file: `data/02_module2/VGR_Masked_TreatmentExercise.csv` (not municipalities)
- Predictive component: **not issued** as a common file. Optional for a group (own model with a Module 2 hold-out, or an external forecast). Otherwise mark **Not assigned**.

If the course team issues a later extract, only `data/` and the manifest change. Your checks stay the same.


---
### Who does what

**You, before the workshop:** bring a **laptop**. Run the issued companion files on your five codes, then join each Verified extract from `data/group_sources/` that your KPI-to-data table named. Formatting and wiring those extracts is your work; there is no source-specific support. Bring a draft five-row decision table that includes **your** approved columns, not only the issued companions.

**The teacher, in the lecture:** explains contracts, joins, missingness, and the safe-stop rule.

**The teacher, in the workshop:** checks the common exercise, releases one defective copy, and checks whether you detected it.

**You, in the workshop:** freeze the baseline, detect the defect, repair it only if justified, otherwise stop safely.

A solution notebook for the demonstration portfolio will be released after the workshop. It is a checkable benchmark, not extra source data for your joins.


---
## Part 1: Load the data contract and the issued files

Treat `Municipality_Code` as an **identifier**, not a measurement.

Two issued tables have different grains:
- **Annual companion:** one row per municipality in year $t_1$. Join on municipality **and** year after confirming both files use the same $t_1$.
- **Static lookup:** one row per municipality. Join on municipality only. Do **not** invent a year column.

The masked file is a teaching exercise. It must not enter the decision table.


In [ ]:
# ==============================================================================
# Load vintage, paths, reference years, and Module 2 companion files
# ==============================================================================
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 16)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

manifest_path = Path("data/manifest.json")
if not manifest_path.exists():
    raise FileNotFoundError(
        "data/manifest.json not found. Run this notebook from the course folder "
        "that contains the data/ directory (see Data_Stream.tex)."
    )

MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
m1 = MANIFEST["module1"]
m2 = MANIFEST["module2"]
T0 = int(m1["t0"])
T1 = int(m1["t1"])
DOCUMENTED_MISSING = list(m2["documented_missing_markers"])
PREDICTIVE_STATUS = m2["predictive_component"]

anchor_path = Path(m1["anchor_file"])
annual_path = Path(m2["annual_file"])
static_path = Path(m2["static_file"])
masked_path = Path(m2["masked_exercise_file"])

print(f"Course vintage        : {MANIFEST['vintage']}")
print(f"Anchor file           : {anchor_path}")
print(f"Annual companion      : {annual_path}")
print(f"Static lookup         : {static_path}")
print(f"Masked exercise       : {masked_path}")
print(f"Reference years       : t0 = {T0}, t1 = {T1}")
print(f"Predictive component  : {PREDICTIVE_STATUS}")
print(f"Documented missing    : {DOCUMENTED_MISSING}")

df_anchor = pd.read_csv(anchor_path)
df_annual_raw = pd.read_csv(annual_path, dtype={"Avg_Travel_Time_min": "string"})
df_static_raw = pd.read_csv(static_path, dtype={"Est_Premises_Cost_kSEK": "string"})
df_masked_raw = pd.read_csv(masked_path, dtype={"Synthetic_Value": "string"})

print(f"\nAnchor rows          : {len(df_anchor)}")
print(f"Annual rows           : {len(df_annual_raw)}")
print(f"Static rows           : {len(df_static_raw)}")
print(f"Masked rows           : {len(df_masked_raw)}")
df_annual_raw.head(3)


---
## Task 1: Confirm the input contracts

Create a short **source register**. For each input, record:

- file identifier and vintage;
- observation unit (what one row represents);
- key fields and whether they must be unique;
- geographic and temporal coverage;
- units and documented missing-value markers;
- which approved indicators the source can support.

Classify each source as:

- annual municipality data;
- a static municipality lookup;
- finer-grained data that would need justified aggregation;
- a scenario or assumption table;
- a synthetic teaching file.

Do not assume that every source should join on both municipality and year.

**Self-check:** Every approved indicator has an identified input route, or an explicitly approved exception.


In [ ]:
# ==============================================================================
# TASK 1: Inspect grain, keys, uniqueness, and missing markers
# ==============================================================================
print("=== Annual companion ===")
print("Columns:", list(df_annual_raw.columns))
print("Year values:", sorted(int(y) for y in df_annual_raw["Year"].unique()))
print("Unique municipality codes:", df_annual_raw["Municipality_Code"].nunique())
print("Duplicate codes:", int(df_annual_raw.duplicated("Municipality_Code").sum()))
print("Rows with documented travel-time marker:")
print(
    df_annual_raw.loc[
        df_annual_raw["Avg_Travel_Time_min"].isin(DOCUMENTED_MISSING),
        ["Municipality_Code", "Municipality_Name", "Avg_Travel_Time_min"],
    ].to_string(index=False)
)

print("\n=== Static lookup ===")
print("Columns:", list(df_static_raw.columns))
print("Has Year column:", "Year" in df_static_raw.columns)
print("Unique municipality codes:", df_static_raw["Municipality_Code"].nunique())
print("Duplicate codes:", int(df_static_raw.duplicated("Municipality_Code").sum()))
print("Rows with documented premises-cost marker:")
print(
    df_static_raw.loc[
        df_static_raw["Est_Premises_Cost_kSEK"].isin(DOCUMENTED_MISSING),
        ["Municipality_Code", "Municipality_Name", "Est_Premises_Cost_kSEK"],
    ].to_string(index=False)
)

print("\n=== Masked teaching file ===")
print(df_masked_raw.to_string(index=False))
print("\nThese records are synthetic. They are not municipalities.")


---
### Your analysis for Task 1

**Source register:**

| Source | Observation unit | Key fields | Coverage | Missing markers | Indicators supplied |
| :--- | :--- | :--- | :--- | :--- | :--- |
|  |  |  |  |  |  |
|  |  |  |  |  |  |
|  |  |  |  |  |  |
|  |  |  |  |  |  |

**Which sources join on municipality and year, and which join on municipality only?**
*(Write here.)*

**Which fields are labelled teaching values rather than independently verified facts?**
*(Write here.)*


---
## Task 2: Standardize keys and execute joins

Complete the common companion-table exercise on **your five allocated municipalities** first. That rehearsal is the method. After it passes, join every additional extract named in your Module 1 KPI-to-data table (Task 2b below).

Your pipeline must:

1. keep unmodified copies of the source inputs (already loaded above);
2. standardize identifiers and data types through code;
3. check key uniqueness **before** joining;
4. declare the intended join relationship;
5. report row counts and unmatched keys;
6. verify that the result is still one row per municipality.

Use a **left** join from the five-municipality $t_1$ extract so alternatives cannot disappear silently. `validate` checks the declared key relationship. `indicator` shows matches and unmatched rows.

A missing source match is a **coverage issue**, not permission to invent a value, and not permission to drop a municipality.

**Self-check:** The audit shows input row counts, output row count, unmatched keys, and join relationship for every merge.


In [ ]:
# ==============================================================================
# TASK 2: Select the portfolio, standardize keys, and join with an audit
# ==============================================================================
# TODO: replace this example list with YOUR allocated codes
portfolio_codes = [1481, 1490, 1463, 1465, 1443]


def standardize_code(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip()


def parse_numeric_with_markers(series: pd.Series, markers) -> pd.Series:
    raw = series.astype("string").str.strip()
    cleaned = raw.mask(raw.isin(list(markers)))
    numeric = pd.to_numeric(cleaned, errors="coerce")
    unexpected = cleaned.notna() & numeric.isna()
    if unexpected.any():
        raise ValueError(
            "Unexpected tokens (not in the documented missing list): "
            + str(cleaned.loc[unexpected].unique().tolist())
        )
    return numeric


demo = df_anchor[
    df_anchor["Municipality_Code"].isin(portfolio_codes) & df_anchor["Year"].eq(T1)
].copy()
hist = df_anchor[
    df_anchor["Municipality_Code"].isin(portfolio_codes) & df_anchor["Year"].eq(T0)
].copy()

for frame in (demo, hist):
    frame["Municipality_Code"] = standardize_code(frame["Municipality_Code"])

annual = df_annual_raw.copy()
annual["Municipality_Code"] = standardize_code(annual["Municipality_Code"])
annual["Avg_Travel_Time_min_num"] = parse_numeric_with_markers(
    annual["Avg_Travel_Time_min"], DOCUMENTED_MISSING
)

static = df_static_raw.copy()
static["Municipality_Code"] = standardize_code(static["Municipality_Code"])
static["Est_Premises_Cost_kSEK_num"] = parse_numeric_with_markers(
    static["Est_Premises_Cost_kSEK"], DOCUMENTED_MISSING
)

portfolio_keys = {str(c) for c in portfolio_codes}
assert set(demo["Municipality_Code"]) == portfolio_keys, "t1 extract is missing a municipality"
assert not demo.duplicated("Municipality_Code").any(), "Duplicate municipality in t1 extract"
assert not annual.duplicated(["Municipality_Code", "Year"]).any(), "Duplicate annual key"
assert not static.duplicated("Municipality_Code").any(), "Duplicate static key"
assert set(annual["Year"].unique()) == {T1}, "Annual Year does not match manifest t1"

annual_cols = [
    "Municipality_Code",
    "Year",
    "Operating_Clinics",
    "Clinics_per_10k",
    "Avg_Travel_Time_min",
    "Avg_Travel_Time_min_num",
]
static_cols = [
    "Municipality_Code",
    "Care_Need_Index",
    "Est_Premises_Cost_kSEK",
    "Est_Premises_Cost_kSEK_num",
]

n_demo, n_annual, n_static = len(demo), len(annual), len(static)

merged_annual = demo.merge(
    annual[annual_cols],
    on=["Municipality_Code", "Year"],
    how="left",
    validate="one_to_one",
    indicator="annual_merge",
)

merged = merged_annual.merge(
    static[static_cols],
    on="Municipality_Code",
    how="left",
    validate="one_to_one",
    indicator="static_merge",
)

print("Join 1 (annual): left t1 extract", n_demo, "x annual", n_annual)
print("  relationship declared: one_to_one on Municipality_Code + Year")
print("  output rows:", len(merged_annual))
print("  merge labels:\n", merged_annual["annual_merge"].value_counts().to_string())
print(
    "  unmatched codes:",
    merged_annual.loc[merged_annual["annual_merge"].eq("left_only"), "Municipality_Code"].tolist(),
)

print("\nJoin 2 (static): left annual result", len(merged_annual), "x static", n_static)
print("  relationship declared: one_to_one on Municipality_Code (no Year on the lookup)")
print("  output rows:", len(merged))
print("  merge labels:\n", merged["static_merge"].value_counts().to_string())
print(
    "  unmatched codes:",
    merged.loc[merged["static_merge"].eq("left_only"), "Municipality_Code"].tolist(),
)

assert len(merged) == 5, "Decision extract is no longer five rows"
assert set(merged["Municipality_Code"]) == portfolio_keys, "An alternative was lost"
print("\nFive alternatives retained:", sorted(merged["Municipality_Code"]))
merged[
    [
        "Municipality_Code",
        "Municipality_Name",
        "Operating_Clinics",
        "Avg_Travel_Time_min",
        "Care_Need_Index",
        "Est_Premises_Cost_kSEK",
        "annual_merge",
        "static_merge",
    ]
].sort_values("Municipality_Code")


---
### Your analysis for Task 2

**What does one row represent after the two joins?**
*(Write here.)*

**Join audit (copy the printed counts):**
- Annual join — input rows, output rows, unmatched keys, relationship:
- Static join — input rows, output rows, unmatched keys, relationship:

**If a municipality had matched the annual file but travel time was `..`, is that an unmatched join or a source missing value? Why must it not become zero?**
*(Write here.)*


---
## Task 2b: Join your Verified KPI extracts

The companion joins above are the **method**. Your architecture is the KPI-to-data table from Module 1.

For every Verified indicator that is not already in the issued companions:

1. keep the original download in `data/group_sources/raw/` (or outside the issued folders);
2. in this notebook, rename keys to `Municipality_Code` (map kommun names to SCB codes if needed);
3. save one analysis-ready CSV per source under `data/group_sources/`;
4. left-join onto the five municipality rows;
5. add a row to `data/group_sources/Source_Register.csv`.

Do **not** edit `data/manifest.json`.

If a source does not cover all five municipalities, keep the five rows, report unmatched codes, and do not score that column in Module 3 until it is finite for all five.


In [ ]:
# ==============================================================================
# TASK 2b: Join each Verified extract from your KPI-to-data table
# ==============================================================================
# Repeat this block once per group CSV. Replace the filename and column list.
# The issued companions stay as they are. Do not put group files in manifest.json.

GROUP_SOURCES = Path("data/group_sources")
print("Files in group_sources:")
if GROUP_SOURCES.exists():
    for p in sorted(GROUP_SOURCES.glob("*.csv")):
        print(" ", p.as_posix())
else:
    print("  (folder missing — create data/group_sources/ and add your extracts)")

# extra = pd.read_csv(GROUP_SOURCES / "your_indicator.csv", dtype={"Municipality_Code": "string"})
# extra["Municipality_Code"] = extra["Municipality_Code"].astype("string").str.strip()
# n_before = len(merged)
# merged = merged.merge(
#     extra,
#     on="Municipality_Code",
#     how="left",
#     indicator=True,
#     validate="one_to_one",
# )
# print(merged["_merge"].value_counts().to_string())
# unmatched = merged.loc[merged["_merge"].eq("left_only"), "Municipality_Code"]
# print("Unmatched municipality codes:", unmatched.tolist())
# merged = merged.drop(columns="_merge")
# assert len(merged) == n_before, "The group-source join changed the number of rows"
# assert set(merged["Municipality_Code"]) == portfolio_keys, "An alternative was lost"

print(
    "Task 2b: uncomment and repeat the merge for each Verified extract. "
    "Skip only if every approved KPI already comes from the issued companions."
)


---
### Your analysis for Task 2b

**Which Verified KPIs did you join here, and from which files?**
*(Write here. If none, say which issued columns already cover the approved set.)*

**Any unmatched codes or grain problems (clinic vs municipality, name vs SCB code)?**
*(Write here.)*


---
## Task 3: Predictive work, if the group takes it up

Prediction is **optional**. The issued common path does not require it. A group may build its own model or use a published forecast from another source. Either way, **justify** the target, the observation unit, and why the figures are predictions.

That work is a **different object** from the five-row decision table.

- If you **build a model**, reserve about 20% of eligible observations for later testing. Fit data-dependent preprocessing on training observations only.
- If you **use an external forecast**, format and join it like any other group source. Do not invent a 20% split of that published table.
- Do not split the five municipalities as if one of them were an MCDA test set.
- Do not split the ten-row Module 1 prototype to create a hold-out file.

The issued manifest marks the common path **Not assigned**. Do not edit that file. If your group takes up prediction, record the justification in this notebook.

**Self-check:** If the group does not take it up, record **Not assigned** and continue.


In [ ]:
# ==============================================================================
# TASK 3: Predictive component — common path is Not assigned; groups may take it up
# ==============================================================================
print("Issued predictive flag (common path):", PREDICTIVE_STATUS)
print("The five-municipality decision table is NOT a modelling sample.")
print("All five alternatives remain in the comparison.")
print("If the group builds a model: hold out about 20% of eligible observations.")
print("If the group uses an external forecast: join it; do not invent a 20% split.")
print("If the group does not take up prediction: mark Not assigned and continue.")


---
### Your analysis for Task 3

**Did the group take up prediction? Own model, external forecast, or Not assigned?**
*(Write here.)*

**If yes, justify the target and source. If you built a model, how did you hold out about 20% of eligible observations?**
*(Write here.)*

**Why should the group's five-municipality decision table not automatically be divided using a training/test rule?**
*(Write here.)*


---
## Task 4: Audit missingness and compare treatments

Create a missingness summary by indicator and municipality.

Distinguish:

- missing values already present in a source;
- missing values caused by unmatched joins;
- not-applicable or suppressed values, where documented;
- values unavailable because coverage differs.

Do **not** treat every missing-value marker as zero. Do **not** drop one of the five alternatives to hide missingness.

### Required treatment comparison

1. Complete the **masked teaching exercise** (mean versus median). Keep it separate from the decision table.
2. Identify one consequential missingness issue in **your** five municipalities, if there is one. Compare two defensible responses and say which one you will use in the decision table. If your five codes have no suitable gap, the masked exercise is your required comparison.

**Self-check:** Show the original affected values, the two treatments, and their effect. State which treatment you selected and why.


In [ ]:
# ==============================================================================
# TASK 4: Missingness audit, masked mean/median rehearsal, portfolio comparison
# ==============================================================================
missing_audit = pd.DataFrame(
    {
        "Municipality_Code": merged["Municipality_Code"],
        "Municipality_Name": merged["Municipality_Name"],
        "Travel_missing_in_source": merged["Avg_Travel_Time_min_num"].isna(),
        "Travel_unmatched_join": merged["annual_merge"].ne("both"),
        "Cost_missing_in_source": merged["Est_Premises_Cost_kSEK_num"].isna(),
        "Cost_unmatched_join": merged["static_merge"].ne("both"),
    }
)
print("Portfolio missingness audit:")
print(missing_audit.to_string(index=False))

# --- Masked teaching exercise (not municipalities; not for the decision table) ---
masked = df_masked_raw.copy()
masked["Value_num"] = parse_numeric_with_markers(masked["Synthetic_Value"], DOCUMENTED_MISSING)
observed = masked.loc[masked["Value_num"].notna(), "Value_num"]
mean_value = float(observed.mean())
median_value = float(observed.median())

masked_compare = pd.DataFrame(
    {
        "Treatment": ["Mean of observed", "Median of observed"],
        "Replacement_for_E5": [mean_value, median_value],
        "Uses_outlier_64": [True, False],
    }
)
print("\nMasked exercise (keep separate from the decision table):")
print(masked.to_string(index=False))
print(masked_compare.to_string(index=False))
print("Observed values:", observed.tolist(), "mean =", mean_value, "median =", median_value)

# --- Portfolio travel-time comparison (Svenljunga in the demonstration set) ---
observed_travel = merged.loc[merged["Avg_Travel_Time_min_num"].notna(), "Avg_Travel_Time_min_num"]
imputed_median = float(observed_travel.median()) if len(observed_travel) else np.nan

travel_compare = merged[
    ["Municipality_Code", "Municipality_Name", "Avg_Travel_Time_min", "Avg_Travel_Time_min_num"]
].copy()
travel_compare["Retain_and_flag"] = travel_compare["Avg_Travel_Time_min_num"]
travel_compare["Impute_portfolio_median"] = travel_compare["Avg_Travel_Time_min_num"].fillna(
    imputed_median
)
print("\nTravel-time treatments (portfolio):")
print(travel_compare.to_string(index=False))
print("Portfolio median of observed travel times:", imputed_median)
print("Selected decision-table treatment: retain the missing value and flag it.")


---
### Your analysis for Task 4

**Missingness classification for your five municipalities:**
*(Write here: source missing, unmatched join, or none.)*

**Masked exercise:** What replacement would E5 receive under mean versus median? Which is more defensible here, and why? Confirm that this file does **not** enter the decision table.
*(Write here.)*

**Portfolio issue:** Which two treatments did you compare, which did you select for the decision table, and what limitation remains?
*(Write here.)*


---
## Task 5: Diagnose preprocessing needs

Inspect the relevant variables for unit or type problems, implausible values, skewness, and incompatible scales **where a later method requires scaling**.

Apply a transformation only when its purpose is clear. A documented decision **not** to transform a variable is acceptable.

- Keep interpretable indicator values in the decision table.
- If you create a transformed copy, give it a new name and retain the original.
- Normalization for MCDA follows the Module 3 method specification. Do **not** apply an arbitrary scaler now merely because the indicators have different units.

Verify the issued derived field:

$$C_{10k} = 10000\,\frac{\text{Operating clinics}}{P_{\mathrm{total},t_1}}$$

**Self-check:** Every implemented treatment has a stated purpose and an inspectable before-and-after comparison or integrity check.


In [ ]:
# ==============================================================================
# TASK 5: Diagnose distributions and verify the derived clinic rate
# ==============================================================================
merged["Clinics_per_10k_check"] = (
    10000.0 * merged["Operating_Clinics"] / merged["Pop_Total"]
)
check_gap = (merged["Clinics_per_10k_check"] - merged["Clinics_per_10k"]).abs().max()
print("Max |issued Clinics_per_10k - recomputed|:", check_gap)

diag_cols = [
    "Pop_Total",
    "Operating_Clinics",
    "Clinics_per_10k",
    "Avg_Travel_Time_min_num",
    "Care_Need_Index",
    "Est_Premises_Cost_kSEK_num",
]
print("\nPortfolio numeric summary (NaNs remain visible):")
print(merged[diag_cols].describe().to_string())

print("\nUnits differ by design (people, clinics, minutes, index, kSEK).")
print("No MCDA scaler is applied in Module 2.")
print("No log transform or winsorization is applied: no purpose has been established.")
print("Svenljunga's missing travel time is a documented gap, not an outlier to cap.")


---
### Your analysis for Task 5

**What did you inspect, and did any value look implausible after checking units and observation meaning?**
*(Write here.)*

**Did you transform any decision-table indicator? If yes, state the purpose. If no, that is an acceptable documented decision.**
*(Write here.)*

**Why is an MCDA scaler not applied in this module?**
*(Write here.)*


---
## Task 6: Produce and verify the five-municipality decision table

Generate:

1. the five-municipality decision table, including issued companion columns **and** columns from your approved `group_sources/` extracts;
2. short indicator metadata (units and reference periods);
3. the join and missingness evidence already printed;
4. the treatment-comparison results already printed;
5. a predictive note: **Not assigned**, or a justified own model / external forecast.

Required checks:

- exactly the five allocated codes, no duplicate municipality rows;
- required indicator columns present;
- missing or assumption-based values explicitly flagged;
- no unexplained row loss or multiplication;
- repeated execution would produce the same outputs.

You are still not choosing a winning municipality.


In [ ]:
# ==============================================================================
# TASK 6: Build the decision table, carry Module 1 indicators, and verify
# ==============================================================================
hist_small = hist[["Municipality_Code", "Pop_80_Plus"]].rename(
    columns={"Pop_80_Plus": "Pop_80_Plus_t0"}
)
decision = merged.merge(hist_small, on="Municipality_Code", how="left", validate="one_to_one")

decision["Share_80_plus_pct"] = 100.0 * decision["Pop_80_Plus"] / decision["Pop_Total"]
decision["Dependency_ratio"] = 100.0 * (
    decision["Pop_0_17"] + decision["Pop_65_79"] + decision["Pop_80_Plus"]
) / decision["Pop_18_64"]
decision["Growth_80_plus_pct"] = 100.0 * (
    decision["Pop_80_Plus"] - decision["Pop_80_Plus_t0"]
) / decision["Pop_80_Plus_t0"]

decision["Travel_time_status"] = np.where(
    decision["Avg_Travel_Time_min_num"].isna(),
    "missing_in_source",
    "observed",
)
decision["Premises_cost_status"] = np.where(
    decision["Est_Premises_Cost_kSEK_num"].isna(),
    "missing_in_source",
    "teaching_estimate",
)
decision["CNI_status"] = "teaching_index"
decision["Clinics_status"] = "teaching_count"
decision["Static_values_reference"] = (
    f"Static lookup reused at t1={T1} (no year on source; assumption labelled)"
)
decision["Predictive_component"] = PREDICTIVE_STATUS

decision_table = decision[
    [
        "Municipality_Code",
        "Municipality_Name",
        "Year",
        "Pop_Total",
        "Share_80_plus_pct",
        "Dependency_ratio",
        "Growth_80_plus_pct",
        "Operating_Clinics",
        "Clinics_per_10k",
        "Avg_Travel_Time_min_num",
        "Travel_time_status",
        "Care_Need_Index",
        "Est_Premises_Cost_kSEK_num",
        "Premises_cost_status",
        "Predictive_component",
    ]
].sort_values("Municipality_Code")

print("Five-municipality decision table:")
print(decision_table.to_string(index=False))

# --- Required output checks ---
codes = set(decision_table["Municipality_Code"])
assert codes == portfolio_keys, f"Code set mismatch: {codes}"
assert len(decision_table) == 5, "Decision table must have five rows"
assert not decision_table.duplicated("Municipality_Code").any(), "Duplicate municipality row"
required_cols = [
    "Share_80_plus_pct",
    "Dependency_ratio",
    "Growth_80_plus_pct",
    "Operating_Clinics",
    "Clinics_per_10k",
    "Avg_Travel_Time_min_num",
    "Care_Need_Index",
    "Est_Premises_Cost_kSEK_num",
    "Travel_time_status",
]
missing_cols = [c for c in required_cols if c not in decision_table.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"
assert decision_table["Travel_time_status"].isin(["observed", "missing_in_source"]).all()
assert (decision_table["Predictive_component"] == PREDICTIVE_STATUS).all()

# Keep columns joined in Task 2b so Module 3 can score YOUR architecture.
reserved = set(decision_table.columns) | {"_merge"}
extra_cols = [c for c in merged.columns if c not in reserved]
if extra_cols:
    decision_table = decision_table.merge(
        merged[["Municipality_Code"] + extra_cols],
        on="Municipality_Code",
        how="left",
        validate="one_to_one",
    )
    print("Added group-collected columns:", extra_cols)
else:
    print("No extra group-collected columns on `merged`. Add them in Task 2b if your KPI table requires them.")

out_path = Path("data/group_sources/decision_table.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
decision_table.to_csv(out_path, index=False)
print("Wrote", out_path.as_posix(), "— load this in Modules 3 and 4 for your architecture.")

print("\nOutput checks: PASS")
print("Row count:", len(decision_table), "| duplicate codes:", int(decision_table.duplicated("Municipality_Code").sum()))

metadata = pd.DataFrame(
    [
        ["Share_80_plus_pct", "percent of population aged 80+", f"t1={T1}", "Module 1 anchor"],
        ["Dependency_ratio", "dependants per 100 aged 18-64", f"t1={T1}", "Module 1 anchor"],
        ["Growth_80_plus_pct", "percent change in population 80+", f"t0={T0} to t1={T1}", "Module 1 anchor"],
        ["Operating_Clinics", "count (teaching value)", f"aligned to t1={T1}", "annual companion"],
        ["Clinics_per_10k", "clinics per 10,000 residents", f"t1={T1}", "derived"],
        ["Avg_Travel_Time_min_num", "minutes; NaN if source marker ..", f"t1={T1}", "annual companion"],
        ["Care_Need_Index", "index (1.0 teaching average)", "static lookup reused at t1", "static companion"],
        ["Est_Premises_Cost_kSEK_num", "thousand SEK (teaching estimate)", "static lookup reused at t1", "static companion"],
    ],
    columns=["Indicator", "Unit_and_meaning", "Reference_period", "Source"],
)
print("\nIndicator metadata:")
print(metadata.to_string(index=False))
decision_table


---
### Your analysis for Task 6

**Does the table contain exactly your five municipalities, and is any value flagged rather than silently filled?**
*(Write here.)*

**What limitation must Module 3 see (coverage gap, teaching-value label, or approved assumption)?**
*(Write here.)*


---
## Live pipeline-integrity challenge (rehearsal)

In class, the instructor releases a controlled copy or patch of the common exercise. The defect may be a duplicate key, an identifier-format inconsistency, an unexpected missing-value marker, a missing required key, or a changed column name.

Your group must:

1. run the existing checks and identify the defect;
2. explain its likely consequence;
3. repair it **only** where a deterministic repair is justified;
4. otherwise **stop safely** and report what information is missing;
5. rerun the affected checks and keep the evidence.

Do not automatically delete duplicates or guess corrected identifiers. A safe stop is correct when the supplied evidence does not support a reliable repair.

This cell rehearses a **duplicate key** on a copy of the companion. Keep the challenge copy separate from approved source data.


In [ ]:
# ==============================================================================
# Live-challenge rehearsal: duplicate key on a copy (not the approved source)
# ==============================================================================
keys = ["Municipality_Code", "Year"]
challenge_annual = pd.concat(
    [annual[annual_cols], annual[annual_cols].iloc[[0]]],
    ignore_index=True,
)

duplicate_rows = challenge_annual.loc[challenge_annual.duplicated(keys, keep=False)]
print("Duplicate key rows in the challenge copy:")
print(duplicate_rows.to_string(index=False))

identical = (
    duplicate_rows.drop(columns=keys).nunique().le(1).all()
    if len(duplicate_rows)
    else False
)
print("\nAre the duplicated records identical on non-key fields?", bool(identical))

try:
    demo.merge(
        challenge_annual,
        on=keys,
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    print("ERROR: the join was allowed. The uniqueness check is too weak.")
except pd.errors.MergeError as err:
    print("\nJoin correctly refused:")
    print(str(err).split("\n")[0])

print("\nAutomatic drop_duplicates() is not the repair.")
print("Inspect first. Identical copies can be collapsed only after that inspection.")
print("Conflicting copies require a safe stop: we cannot tell which row is authoritative.")

# Conflicting-duplicate demonstration (still a copy):
conflict = challenge_annual.copy()
conflict.loc[conflict.index[-1], "Operating_Clinics"] = -99
conflict_dups = conflict.loc[conflict.duplicated(keys, keep=False)]
print("\nIf the duplicate values conflict:")
print(conflict_dups[keys + ["Operating_Clinics"]].to_string(index=False))
print("Safe stop: do not guess which clinic count is correct; do not proceed to the decision table from this copy.")


---
### Your analysis for the challenge rehearsal

**What defect did the checks catch?**
*(Write here.)*

**What would happen if the join proceeded without `validate`?**
*(Write here.)*

**What action is justified here: a documented repair, or a safe stop? Why?**
*(Write here.)*


---
## Gateway 2 closing check

Before submitting, confirm that the group can say:

> We know what each row and indicator represents, how the sources were combined, what data issues remain, why our treatments are justified, and how to reproduce and check the outputs.

Bring to Workshop 2:

- this notebook, run on the common companion files;
- your source register and join/missingness prints;
- the five-row decision table with flags;
- unresolved questions (including any Module 1 corrections still outstanding).

Gateway 2 also requires a two-page audit report (outside this notebook) summarizing the issues that matter for the establishment analysis. Detailed tables stay in the notebook and appendices.

You still have not selected a winning municipality. Module 3 uses this table to compare alternatives.
